# DIY Copilot
## Lesson: Context Engineering

Everything a model knows about your task is in the **context window** of a single request. The model is stateless: with every request the harness sends the complete context again – the system prompt, the tool declarations, the conversation so far, every tool result:

```
┌──────────────────────────────────────── one request ─────────────────────────────────────────┐
│ system prompt │ tool declarations │ instructions, skills, code │ conversation: user, assistant, │
│               │                   │                            │ tool calls, tool results, ...  │
└──────────────────────────────────────────────────────────────────────────────────────────────┘
  stable: the same in every request ──────────────────────────►   grows with every step ───────►
```

**Context engineering** is the design of this context: *what* goes in, *when*, in *which order* – and what stays out. Prompt engineering is about the wording; context engineering treats the context window as a limited resource of the harness. More context is not better: every token costs money and latency, and with long, noisy contexts the answers get worse (*context rot*). That's why coding assistants such as Claude Code, GitHub Copilot or Cursor put a lot of effort into it.

During this lesson you will learn:
* where the tokens of a coding session go (and why the costs grow faster than the conversation)
* how **prompt caching** makes the stable part cheaper – and how a harness can break it
* how **progressive disclosure** keeps the stable part small: the assistant gets an index of **skills** and loads their instructions only when it needs them
* how **history condensing** keeps the growing part small: clearing old tool results and summarizing the conversation

>**Free tier:** This notebook makes about 20 LLM requests and no embedding requests.

### Set up the environment

In [ ]:
# >>> SETUP <<<
# In Google Colab, this cell
#   1. clones the workshop repository,
#   2. installs its dependencies into the Python environment of this notebook (with uv),
#   3. makes the shared code (lib) importable.
# Locally (after `uv sync` in the repository), only step 3 is needed.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/openknowledge/diy-copilot-bootcamp"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/diy-copilot")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, str(REPO_DIR / "setup" / "install_dependencies.py")], check=True)
else:
    # The repository root is the first parent directory containing the shared code
    REPO_DIR = next(p for p in Path.cwd().resolve().parents if (p / "lib").is_dir() and (p / "pyproject.toml").exists())

# The repository is not an installed package, so we add it to the module search path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Workshop repository: {REPO_DIR}")

In [ ]:
import json
import re
from datetime import datetime
from pathlib import Path

import litellm
import pandas as pd
import yaml
from litellm import completion, completion_cost, token_counter

from lib.constants import CODEBASE_PATH, SKILLS_PATH
from lib.environment import load_environment
from lib.retry import call_with_retry

default_models = load_environment()
generation_model = default_models["generation_model"]
print(f"Generation model: {generation_model}")


def count_tokens(text: str) -> int:
    """Estimates the number of tokens of a text.

    LiteLLM counts locally with a generic tokenizer. For Gemini it underestimates by about 20% -
    good enough to compare variants. The exact numbers are in `response.usage`.
    """
    return token_counter(model=generation_model, text=text)


def print_usage(label: str, response) -> None:
    """Prints the token usage and the costs of a response (cached tokens are cheaper)."""
    usage = response.usage
    cached = (usage.prompt_tokens_details.cached_tokens if usage.prompt_tokens_details else None) or 0
    try:
        costs = f"{completion_cost(completion_response=response):.6f} USD"
    except Exception:
        costs = "unknown"
    print(f"{label:28} prompt tokens: {usage.prompt_tokens:6}   cached: {cached:6}   costs: {costs}")

### The coding session we use in this lesson

A developer of Team Workplace fixes the bug `WORK-412` in Roomie (`500 Internal Server Error` when a booking is created without timezone) with the help of the assistant. The assistant has a tool `read_file`, which you know from the function calling lesson.

We don't run this session live – that would cost a lot of requests. Instead, we build the conversation as a list of messages, exactly as the harness would have collected it: the developer pastes a server log, they make a decision together, the assistant reads three files, the developer pastes a test output. Read through the messages to know the story.

In [ ]:
def read_file(path: str) -> str:
    """Reads a file of the Roomie code base."""
    return (CODEBASE_PATH / path).read_text()


read_file_tool = {
    "type": "function",
    "function": {
        "name": "read_file",
        "description": "Reads a file of the Roomie code base and returns its content.",
        "parameters": {
            "type": "object",
            "properties": {"path": {"type": "string", "description": "Path relative to the repository root, e.g. 'roomie/models.py'."}},
            "required": ["path"],
        },
    },
}


def tool_call(call_id: str, name: str, **arguments) -> dict:
    """An assistant message that requests a tool call."""
    return {"role": "assistant", "content": None, "tool_calls": [{"id": call_id, "type": "function", "function": {"name": name, "arguments": json.dumps(arguments)}}]}


def tool_result(call_id: str, content: str) -> dict:
    """The tool message with the result of a tool call."""
    return {"role": "tool", "tool_call_id": call_id, "content": content}


# A server log as developers paste it: the same traceback for every failing request
TRACEBACK = """Traceback (most recent call last):
  File "/app/.venv/lib/python3.12/site-packages/starlette/middleware/errors.py", line 165, in __call__
    await self.app(scope, receive, _send)
  File "/app/.venv/lib/python3.12/site-packages/starlette/middleware/exceptions.py", line 62, in __call__
    await wrap_app_handling_exceptions(self.app, conn)(scope, receive, send)
  File "/app/.venv/lib/python3.12/site-packages/starlette/routing.py", line 715, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/app/.venv/lib/python3.12/site-packages/starlette/routing.py", line 735, in app
    await route.handle(scope, receive, send)
  File "/app/.venv/lib/python3.12/site-packages/starlette/routing.py", line 288, in handle
    await self.app(scope, receive, send)
  File "/app/.venv/lib/python3.12/site-packages/fastapi/routing.py", line 301, in app
    raw_response = await run_endpoint_function(dependant=dependant, values=values, is_coroutine=is_coroutine)
  File "/app/.venv/lib/python3.12/site-packages/fastapi/routing.py", line 214, in run_endpoint_function
    return await run_in_threadpool(dependant.call, **values)
  File "/app/roomie/routers/bookings.py", line 20, in create_booking
    return service.create_booking(request, user)
  File "/app/roomie/services/booking_service.py", line 36, in create_booking
    validate_new_booking(request, room, existing, now)
  File "/app/roomie/services/booking_rules.py", line 111, in validate_new_booking
    check_advance_window(request.start, now)
  File "/app/roomie/services/booking_rules.py", line 61, in check_advance_window
    if start <= now:
TypeError: can't compare offset-naive and offset-aware datetimes"""
SERVER_LOG = "\n".join(
    f"2026-03-03 10:{minute:02d}:14,118 INFO  roomie.access request_id=req-{minute:04x} GET /api/v2/rooms/availability -> 200\n"
    f"2026-03-03 10:{minute:02d}:31,502 ERROR roomie.main request_id=req-{minute + 100:04x} POST /api/v2/bookings -> 500\n{TRACEBACK}"
    for minute in range(10, 22)
)

PYTEST_OUTPUT = """============================= test session starts ==============================
platform linux -- Python 3.12.7, pytest-8.4.2, pluggy-1.6.0
rootdir: /home/dev/roomie
configfile: pyproject.toml
collected 23 items

tests/test_booking_rules.py ..........                                    [ 43%]
tests/test_bookings_api.py ........F                                     [ 82%]
tests/test_rooms_api.py ....                                             [100%]

=================================== FAILURES ===================================
__________________ test_booking_without_timezone_is_rejected ___________________

client = <starlette.testclient.TestClient object at 0x7f3a2c1b9d90>

    def test_booking_without_timezone_is_rejected(client):
        \"\"\"Regression test for WORK-412.\"\"\"
        payload = booking_payload()
        payload["start"] = "2026-03-03T14:00:00"
        payload["end"] = "2026-03-03T15:00:00"
>       response = client.post("/api/v2/bookings", json=payload, headers=ALICE)

tests/test_bookings_api.py:76:
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
""" + TRACEBACK.split("\n", 1)[1] + """
=========================== short test summary info ============================
FAILED tests/test_bookings_api.py::test_booking_without_timezone_is_rejected - TypeError: can't compare offset-naive and offset-aware datetimes
========================= 1 failed, 22 passed in 0.84s ========================="""

SESSION_SYSTEM_PROMPT = (
    "You are a coding assistant for Team Workplace, the team that owns Roomie, the meeting room booking service. "
    "Use the read_file tool to look at the code. Follow the company guidelines and keep your answers short."
)

session = [
    {"role": "system", "content": SESSION_SYSTEM_PROMPT},
    {"role": "user", "content": f"The mobile app team reports 500 errors when they create bookings (WORK-412). Here is the server log:\n```\n{SERVER_LOG}\n```"},
    {"role": "assistant", "content": (
        "All failing requests end with `TypeError: can't compare offset-naive and offset-aware datetimes` in `check_advance_window` "
        "(`roomie/services/booking_rules.py`). The mobile app sends `start`/`end` without offset, e.g. `2026-03-03T10:00:00`. "
        "Pydantic parses them as naive datetimes, and comparing them with the timezone-aware `now` raises the error. "
        "According to API-07, such timestamps must be rejected with a 422 - a 500 is always a bug (API-06).")},
    {"role": "user", "content": "Could we simply assume Berlin time for timestamps without offset? The mobile team would like that."},
    {"role": "assistant", "content": (
        "Technically yes, but I would not do it: API-07 requires rejecting timestamps without offset, guessing hides client bugs, "
        "and around the daylight saving time switch local times are ambiguous (see ADR-0004 and WORK-389). "
        "I recommend rejecting them with a 422 and a dedicated error code, so that clients can show a clear message.")},
    {"role": "user", "content": (
        "Agreed. Decision: we reject timestamps without offset with the new error code ROOM-422-NAIVE-TIME "
        "(exception class NaiveTimestampError). We don't guess the timezone. The check becomes the first rule in "
        "validate_new_booking, in a new function check_timezone_aware. The mobile team agreed to always send offsets.")},
    {"role": "assistant", "content": "Got it: new rule `check_timezone_aware` as the first rule in `validate_new_booking`, raising `NaiveTimestampError` with the code `ROOM-422-NAIVE-TIME`. Let's look at the code."},
    {"role": "user", "content": "Show me the booking rules."},
    tool_call("call_1", "read_file", path="roomie/services/booking_rules.py"),
    tool_result("call_1", read_file("roomie/services/booking_rules.py")),
    {"role": "assistant", "content": "`validate_new_booking` runs six rules in a fixed order: slot alignment, duration, business hours, advance window, capacity and overlap. `check_advance_window` is the first rule that compares with `now` - that's where it crashes."},
    {"role": "user", "content": "How are the errors defined?"},
    tool_call("call_2", "read_file", path="roomie/errors.py"),
    tool_result("call_2", read_file("roomie/errors.py")),
    {"role": "assistant", "content": "Every error is a subclass of `RoomieError` with a `status_code` and a stable `code`. A central handler converts them into `{\"error\": {\"code\": ..., \"message\": ...}}`."},
    {"role": "user", "content": "And how are the API tests set up?"},
    tool_call("call_3", "read_file", path="tests/conftest.py"),
    tool_result("call_3", read_file("tests/conftest.py")),
    {"role": "assistant", "content": "The `client` fixture creates the app with in-memory fakes and a `FakeClock` frozen on Monday, 2 March 2026, 09:00 Berlin time. `booking_payload()` and `at()` build requests; `ALICE`, `BOB` and `FIONA` are the API keys."},
    {"role": "user", "content": f"I wrote the regression test. It fails as expected:\n```\n{PYTEST_OUTPUT}\n```"},
    {"role": "assistant", "content": "Good - the test reproduces the bug (TEST-07): it fails with the `TypeError`. After the fix it should get a 422 with the new error code."},
    {"role": "user", "content": "Unrelated question: how long do we keep application logs?"},
    {"role": "assistant", "content": "30 days in the central log platform (SEC-05, DP-03)."},
]

for message in session:
    content = message["content"] or f"(tool call: {message['tool_calls'][0]['function']['name']}{message['tool_calls'][0]['function']['arguments']})"
    print(f"{message['role']:10} {content[:110].replace(chr(10), ' ')}")

### Example: Where do the tokens go?

Let's measure the session. First: which parts of the context use the most tokens?

In [ ]:
def message_text(message: dict) -> str:
    """The text of a message as it is sent to the model (content or tool call)."""
    if message.get("tool_calls"):
        return json.dumps(message["tool_calls"])
    return message["content"] or ""


def context_category(message: dict) -> str:
    if message["role"] == "tool":
        return "tool results"
    if message["role"] == "user" and len(message["content"]) > 1000:
        return "pasted logs and test output"
    return {"system": "system prompt", "user": "developer messages", "assistant": "assistant messages"}[message["role"]]


tokens = pd.DataFrame(
    [{"part": context_category(m), "tokens": count_tokens(message_text(m))} for m in session]
    + [{"part": "tool declarations", "tokens": count_tokens(json.dumps([read_file_tool]))}]
)
tokens.groupby("part")["tokens"].sum().sort_values(ascending=False).to_frame()

The developer's questions and the assistant's answers are tiny. Almost all tokens are **material**: pasted logs, test output, file contents.

Second: the harness sends a request after every developer message *and* after every tool result – each time with everything that came before. How many tokens does it send in total?

In [ ]:
requests = []
sent_so_far = 0
for index, message in enumerate(session):
    if message["role"] in ("user", "tool"):  # after these messages the harness calls the model
        prompt_tokens = sum(count_tokens(message_text(m)) for m in session[: index + 1]) + count_tokens(json.dumps([read_file_tool]))
        sent_so_far += prompt_tokens
        requests.append({"request": len(requests) + 1, "after": f"{message['role']}: {message_text(message)[:40]}", "prompt tokens": prompt_tokens, "sent in total": sent_so_far})

pd.DataFrame(requests).set_index("request")

What did we learn?
* Every request resends everything before it. The tokens sent in total grow **quadratically** with the length of the session – the server log from the first message is paid for in every single request.
* There are three levers, and each one is an exercise of this lesson:
  1. Make the **stable part** cheaper → *prompt caching* (exercise 01)
  2. Make the **stable part** smaller → *progressive disclosure* (exercise 02)
  3. Keep the **growing part** small → *history condensing* (exercise 03)

### Exercise 01: Prompt caching

To process a prompt, the model computes internal states for every token. If many requests start with exactly the same tokens, the provider can store these states and reuse them: **prompt caching**. Cached tokens are much cheaper (with Gemini: 10% of the normal price) and the request can be faster.

Gemini caches *implicitly*: you don't have to do anything, it only reports in `usage.prompt_tokens_details.cached_tokens` how many tokens came from the cache. Other providers work similarly (OpenAI caches automatically, Anthropic needs explicit cache markers in the request). The rules are always the same:

* Only an identical **prefix** is cached. A single changed token at the beginning invalidates everything after it.
* The prefix must have a minimum length (a few thousand tokens), and the cache expires after a few minutes without use.
* The first request with a new prefix is always a miss – it fills the cache. Caching is best effort, not guaranteed.

Coding assistants make heavy use of it: the system prompt, the tool declarations and the project instructions are the same in every request of a session. Let's put the whole Roomie code base into the system prompt (about 12,000 tokens) and ask two different questions.

In [ ]:
code_base = "\n\n".join(
    f"# File: {path.relative_to(CODEBASE_PATH)}\n{path.read_text()}"
    for path in sorted(CODEBASE_PATH.rglob("*.py"))
    if "__pycache__" not in path.parts
)
ASSISTANT_INSTRUCTIONS = (
    "You are a coding assistant for Team Workplace, the team that owns Roomie, the meeting room booking service. "
    "Answer questions about the code below. Keep your answers short."
)
SYSTEM_PROMPT_WITH_CODE = f"{ASSISTANT_INSTRUCTIONS}\n\n## Code base\n\n{code_base}"
print(f"System prompt: about {count_tokens(SYSTEM_PROMPT_WITH_CODE)} tokens")


def ask(messages: list[dict]):
    return call_with_retry(completion, model=generation_model, messages=messages, temperature=0.0, max_tokens=100)


for question in ["Which function checks the business hours?", "Where is the API key of a request checked?"]:
    response = ask([{"role": "system", "content": SYSTEM_PROMPT_WITH_CODE}, {"role": "user", "content": question}])
    print_usage(question[:28], response)

The second request should report about 10,000 cached tokens. (If the first one does too, somebody – maybe you – used the same prefix a few minutes ago.)

Many harnesses add **dynamic information** to the system prompt: the current date, the name of the developer, the current git branch. The function below puts it at the very beginning. Run the cell: what happens to the cache?

In [ ]:
def build_messages_naive(question: str, developer: str, branch: str) -> list[dict]:
    """Builds the request with session information at the beginning of the system prompt."""
    session_info = f"Current time: {datetime.now():%Y-%m-%d %H:%M:%S}\nDeveloper: {developer}\nGit branch: {branch}\n\n"
    return [
        {"role": "system", "content": session_info + SYSTEM_PROMPT_WITH_CODE},
        {"role": "user", "content": question},
    ]


for question in ["Which function checks the business hours?", "Where is the API key of a request checked?"]:
    response = ask(build_messages_naive(question, developer="alice", branch="fix/WORK-412-naive-datetimes"))
    print_usage(question[:28], response)

The time changes with every request – and because it comes first, the prefix is different every time. Not a single token is cached.

Your task is to build the request so that the same information reaches the model, but the cache works again.

>**Note:** The first request with your new prefix fills the cache. Expect a hit from the second request on.

In [ ]:
# TODO: Build the request so that the stable part comes first and the dynamic session information last
def build_messages(question: str, developer: str, branch: str) -> list[dict]:
    """Builds the request with a stable system prompt; session information goes into the user message."""
    session_info = f"Current time: {datetime.now():%Y-%m-%d %H:%M:%S}\nDeveloper: {developer}\nGit branch: {branch}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT_WITH_CODE},
        {"role": "user", "content": f"{session_info}\n\n{question}"},
    ]


for question in ["Which function checks the business hours?", "Where is the API key of a request checked?"]:
    response = ask(build_messages(question, developer="alice", branch="fix/WORK-412-naive-datetimes"))
    print_usage(question[:28], response)

What does this mean for a team? Let's extrapolate the costs with the token numbers of the last response – as in the augmentation lesson: 1,000 requests per developer and day, a team of 10 developers.

In [ ]:
model_info = litellm.get_model_info(generation_model)
prompt_tokens = response.usage.prompt_tokens
cached_tokens = response.usage.prompt_tokens_details.cached_tokens or 0
requests_per_day = 1_000 * 10

costs_without_cache = prompt_tokens * model_info["input_cost_per_token"]
costs_with_cache = (prompt_tokens - cached_tokens) * model_info["input_cost_per_token"] + cached_tokens * model_info["cache_read_input_token_cost"]

print(f"Input costs per request without cache: {costs_without_cache:.6f} USD")
print(f"Input costs per request with cache:    {costs_with_cache:.6f} USD")
print(f"Per day for the team: {costs_without_cache * requests_per_day:.2f} USD vs. {costs_with_cache * requests_per_day:.2f} USD")

What did we learn?
* Order the context from **stable to dynamic**: system prompt, tool declarations, instructions and code first; session information, conversation and the current question last.
* Things that break the cache: timestamps or ids at the beginning, tool declarations in a different order, editing earlier messages of the conversation. Appending to the conversation keeps the cache – that's why the tool loop is cache friendly.
* Caching makes the context cheaper, but not smaller: the model still has to read all 12,000 tokens – and its attention is spread over all of them.

### Exercise 02: Progressive disclosure with skills

Teams have many recurring tasks with their own conventions: commit messages, merge requests, regression tests, security reviews. A coding assistant should follow them – but putting all instructions into every request would make the stable part big and noisy.

**Skills** solve this with *progressive disclosure*: the assistant first sees only a small index and loads details when it needs them. A skill is a folder with a `SKILL.md` file in the open [Agent Skills](https://agentskills.io) format – a YAML header with `name` and `description`, followed by the instructions:

```
data/skills/
├── write-commit-message/
│   └── SKILL.md
├── write-merge-request-description/
│   ├── SKILL.md
│   └── assets/template.md
└── ...
```

| Level | What | When is it in the context? |
|---|---|---|
| 1 – index | `name` and `description` of every skill | always (system prompt) |
| 2 – instructions | the body of `SKILL.md` | when the model calls `load_skill` |
| 3 – resources | templates, references, scripts of a skill | when the instructions refer to them |

This is the same idea as *just-in-time retrieval* in the function calling lesson: the model decides *when* it needs knowledge. Let's look at the skills of Team Workplace.

In [ ]:
def read_skill(skill_dir: Path) -> dict:
    """Reads the SKILL.md file of a skill: YAML frontmatter between '---' lines, followed by the instructions."""
    _, frontmatter, body = (skill_dir / "SKILL.md").read_text().split("---", 2)
    metadata = yaml.safe_load(frontmatter)
    return {"name": metadata["name"], "description": metadata["description"], "instructions": body.strip(), "path": skill_dir}


skills = {skill["name"]: skill for skill in (read_skill(d) for d in sorted(SKILLS_PATH.iterdir()) if (d / "SKILL.md").exists())}

for skill in skills.values():
    print(f"{skill['name']:34} {count_tokens(skill['instructions']):5} tokens")
print()
print((SKILLS_PATH / "write-commit-message" / "SKILL.md").read_text())

Your first task is to build the **index** (level 1): one line per skill with its name and description. Compare its size with putting all instructions into the system prompt.

In [ ]:
# TODO: Build the skill index - one line per skill with name and description
def build_skill_index(skills: dict) -> str:
    return "\n".join(f"- {skill['name']}: {skill['description']}" for skill in skills.values())


skill_index = build_skill_index(skills)
all_instructions = "\n\n".join(skill["instructions"] for skill in skills.values())

print(skill_index)
print()
print(f"Index:            about {count_tokens(skill_index)} tokens")
print(f"All instructions: about {count_tokens(all_instructions)} tokens")

Your next task is to implement level 2: the function `load_skill` and its tool declaration. Make sure that

* the result tells the model which **additional files** the skill has (for level 3),
* an unknown skill name returns an error message instead of raising an exception – the model can correct itself (remember `execute_tool_call` from the function calling lesson).

>**Hint:** A JSON schema can restrict a string parameter to a list of values with `"enum"`.

In [ ]:
# TODO: Implement load_skill and its tool declaration
def load_skill(name: str) -> str:
    """Returns the instructions of a skill and the list of its additional files."""
    if name not in skills:
        return f"Error: unknown skill '{name}'. Available skills: {list(skills)}"
    skill = skills[name]
    files = sorted(str(p.relative_to(skill["path"])) for p in skill["path"].rglob("*") if p.is_file() and p.name != "SKILL.md")
    result = f"# Skill: {name}\n\n{skill['instructions']}"
    if files:
        result += f"\n\nAdditional files of this skill (read them with read_skill_file when the instructions refer to them): {files}"
    return result


load_skill_tool = {
    "type": "function",
    "function": {
        "name": "load_skill",
        "description": "Loads the full instructions of a skill. Call it before you work on a task that matches the description of a skill.",
        "parameters": {
            "type": "object",
            "properties": {"name": {"type": "string", "enum": list(skills), "description": "The name of the skill."}},
            "required": ["name"],
        },
    },
}

print(load_skill("write-adr")[-300:])

Now we put everything together. The system prompt contains the index and tells the model how to use it. For the tool loop we reuse `execute_tool_call` and `run_assistant` from the function calling lesson (slightly extended: the system prompt is a parameter now).

In [ ]:
SKILLS_SYSTEM_PROMPT = f"""You are a coding assistant for Team Workplace, the team that owns Roomie, the meeting room booking service.

## Skills
Skills are instructions for recurring tasks of the team. Before you start a task, check whether a skill matches it.
If one does, call load_skill first and follow its instructions exactly. Don't load skills you don't need.

{skill_index}"""


def execute_tool_call(tool_call, tools: dict) -> str:
    """Executes a tool call and always returns a string for the model - also in case of errors."""
    name = tool_call.function.name
    if name not in tools:
        return f"Error: unknown tool '{name}'. Available tools: {list(tools)}"
    try:
        arguments = json.loads(tool_call.function.arguments)
    except json.JSONDecodeError:
        return "Error: the arguments are not valid JSON."
    try:
        return str(tools[name](**arguments))
    except Exception as error:
        return f"Error while executing {name}: {type(error).__name__}: {error}"


def run_assistant(system_prompt: str, question: str, tools: dict, tool_declarations: list[dict], max_steps: int = 5) -> str:
    """Answers a question, letting the model call tools as often as it needs (up to max_steps)."""
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        response = call_with_retry(completion, model=generation_model, messages=messages, tools=tool_declarations, temperature=0.0)
        message = response.choices[0].message
        messages.append(message.model_dump())
        if not message.tool_calls:
            return message.content
        for tool_call in message.tool_calls:
            print(f"[step {step}] {tool_call.function.name}({tool_call.function.arguments})")
            result = execute_tool_call(tool_call, tools)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
    return "I could not finish the task within the allowed number of steps."

The developer from our session has fixed `WORK-412` and asks for a commit message. The skill `write-commit-message` contains a convention that the model can't know without it: the `Assisted-by:` trailer.

In [ ]:
WORK_412_DIFF = '''diff --git a/roomie/errors.py b/roomie/errors.py
@@ -70,6 +70,11 @@ class SlotAlignmentError(RoomieError):
+class NaiveTimestampError(RoomieError):
+    status_code = 422
+    code = "ROOM-422-NAIVE-TIME"
+
+
 class DurationError(RoomieError):
diff --git a/roomie/services/booking_rules.py b/roomie/services/booking_rules.py
@@ -21,6 +22,15 @@ from roomie.models import Booking, BookingCreate, Room
+def check_timezone_aware(start: datetime, end: datetime) -> None:
+    """Start and end must have a timezone offset (API-07, ADR-0004)."""
+    for moment in (start, end):
+        if moment.tzinfo is None or moment.utcoffset() is None:
+            raise NaiveTimestampError(
+                "Timestamps must include a timezone offset, e.g. 2026-03-03T10:00:00+01:00."
+            )
+
+
@@ -106,6 +116,7 @@ def validate_new_booking(
     """Runs all booking rules in order. Raises the first violation."""
+    check_timezone_aware(request.start, request.end)
     check_slot_alignment(request.start, request.end)
diff --git a/tests/test_bookings_api.py b/tests/test_bookings_api.py
@@ -68,3 +68,12 @@ def test_check_in_within_grace_period(client, clock):
+
+def test_booking_without_timezone_is_rejected(client):
+    """Regression test for WORK-412."""
+    payload = booking_payload()
+    payload["start"] = "2026-03-03T14:00:00"
+    payload["end"] = "2026-03-03T15:00:00"
+    response = client.post("/api/v2/bookings", json=payload, headers=ALICE)
+    assert response.status_code == 422
+    assert response.json()["error"]["code"] == "ROOM-422-NAIVE-TIME"
'''

skill_tools = {"load_skill": load_skill}
answer = run_assistant(
    SKILLS_SYSTEM_PROMPT,
    f"I fixed WORK-412. Please write the commit message for this diff:\n```diff\n{WORK_412_DIFF}```",
    tools=skill_tools,
    tool_declarations=[load_skill_tool],
)
print(f"\n{answer}\n")

checks = {
    "Conventional Commit header 'fix(<scope>): ...'": re.search(r"^fix\([a-z-]+\): ", answer, re.MULTILINE),
    "Ticket reference 'Refs: WORK-412'": "Refs: WORK-412" in answer,
    "Trailer 'Assisted-by:' (only in the skill)": "Assisted-by:" in answer,
}
for check, passed in checks.items():
    print(f"{'✅' if passed else '❌'} {check}")

And a question that doesn't need a skill – the model should answer directly, without loading anything:

In [ ]:
answer = run_assistant(SKILLS_SYSTEM_PROMPT, "What does the HTTP status code 410 mean? One sentence.", tools=skill_tools, tool_declarations=[load_skill_tool])
print(answer)

### (Optional) Exercise 02b: Level 3 – files of a skill

The skill `write-merge-request-description` tells the model to read its template `assets/template.md`. Your task is to implement the tool `read_skill_file(skill, path)`.

>**Careful:** The path comes from the model. Make sure it can only read files *inside* the skill folder – a path like `../../../.env` must not work. (Remember the guardrails lesson: a manipulated model will try exactly that.)

In [ ]:
# TODO: Implement read_skill_file and its tool declaration
def read_skill_file(skill: str, path: str) -> str:
    """Returns the content of an additional file of a skill."""
    if skill not in skills:
        return f"Error: unknown skill '{skill}'. Available skills: {list(skills)}"
    skill_dir = skills[skill]["path"].resolve()
    file = (skill_dir / path).resolve()
    if not file.is_relative_to(skill_dir) or not file.is_file():
        return f"Error: '{path}' is not a file of the skill '{skill}'."
    return file.read_text()


read_skill_file_tool = {
    "type": "function",
    "function": {
        "name": "read_skill_file",
        "description": "Reads an additional file of a skill, e.g. a template. Only use it when the instructions of a loaded skill refer to the file.",
        "parameters": {
            "type": "object",
            "properties": {
                "skill": {"type": "string", "enum": list(skills), "description": "The name of the skill."},
                "path": {"type": "string", "description": "Path of the file inside the skill folder, e.g. 'assets/template.md'."},
            },
            "required": ["skill", "path"],
        },
    },
}

# Test the protection - no request needed
print(read_skill_file("write-adr", "../../../.env"))

In [ ]:
answer = run_assistant(
    SKILLS_SYSTEM_PROMPT,
    f"Please write the merge request description for my fix of WORK-412:\n```diff\n{WORK_412_DIFF}```",
    tools={"load_skill": load_skill, "read_skill_file": read_skill_file},
    tool_declarations=[load_skill_tool, read_skill_file_tool],
)
print(f"\n{answer}\n")
print(f"{'✅' if 'Required approvals' in answer else '❌'} Uses the template of the skill (section 'Required approvals')")

What did we learn?
* The skill index costs a few hundred tokens instead of a few thousand – and it stays the same in every request, so it is cached, too. The instructions arrive as a tool result *at the end* of the context, only when needed.
* The `description` decides whether a skill is used – just like the description of a tool. Write it like a prompt: *what* the skill does and *when* to use it.
* Skills are just files. The same folders work in other harnesses that support the Agent Skills format (e.g. Claude Code, GitHub Copilot) – like an MCP server, one skill can serve many assistants.
* Everything the model reads – also skills from a shared repository – is part of the context and can contain injected instructions. Review skills like code.

### Exercise 03: History condensing

Back to our session. Every request sends the complete conversation – including the server log, three files and the test output. At some point the session reaches the token budget of the harness (or the context window of the model). Then the harness has to shrink the history. There are three common strategies:

| Strategy | How | Costs | Risk |
|---|---|---|---|
| Sliding window | drop the oldest turns | free | important information from the beginning is lost |
| Clear tool results | replace old tool results with a placeholder | free | the model has to call the tool again if it needs the result |
| Summarize | let the model write a summary of the old turns | one request | details that the summary leaves out are lost |

To test them, the developer asks a follow-up question that needs the **decision** from the beginning of the session:

In [ ]:
FOLLOW_UP = "Now write the new exception class for roomie/errors.py and the new row for docs/error_codes.md for the error we agreed on."


def history_tokens(messages: list[dict]) -> int:
    return sum(count_tokens(message_text(m)) for m in messages)


def ask_follow_up(label: str, messages: list[dict], max_steps: int = 3) -> None:
    """Sends the follow-up question with the given history (in a small tool loop) and checks the answer."""
    messages = messages + [{"role": "user", "content": FOLLOW_UP}]
    prompt_tokens, costs = 0, 0.0
    for step in range(1, max_steps + 1):
        response = call_with_retry(completion, model=generation_model, messages=messages, tools=[read_file_tool], temperature=0.0)
        prompt_tokens += response.usage.prompt_tokens
        costs += completion_cost(completion_response=response)
        message = response.choices[0].message
        messages.append(message.model_dump())
        if not message.tool_calls:
            break
        for tool_call in message.tool_calls:
            print(f"[step {step}] {tool_call.function.name}({tool_call.function.arguments})")
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": execute_tool_call(tool_call, {"read_file": read_file})})
    answer = message.content or ""
    print(f"\n{answer}\n")
    print(f"{label}: {step} request(s), {prompt_tokens} prompt tokens, {costs:.6f} USD")
    print(f"{'✅' if 'ROOM-422-NAIVE-TIME' in answer else '❌'} Uses the agreed error code ROOM-422-NAIVE-TIME")


print(f"Full history: about {history_tokens(session)} tokens")

#### Sliding window

The simplest strategy keeps the system prompt and the last *n* turns. A turn starts with a message of the developer – we never cut between a tool call and its result, the provider would reject the request.

In [ ]:
def turn_starts(messages: list[dict]) -> list[int]:
    """Indices of the messages where a turn starts (messages of the developer)."""
    return [index for index, message in enumerate(messages) if message["role"] == "user"]


def keep_last_turns(messages: list[dict], turns: int) -> list[dict]:
    """Sliding window: the system prompt and the last turns of the conversation."""
    first_kept = turn_starts(messages)[-turns]
    return [messages[0]] + messages[first_kept:]


window = keep_last_turns(session, turns=3)
print(f"Sliding window: about {history_tokens(window)} tokens\n")
ask_follow_up("Sliding window", window)

The history is small – but the decision is gone. The model may read `roomie/errors.py` again, but the agreed code isn't in any file: it invents an error code that looks plausible, but nobody agreed on it.

#### Clear old tool results

Tool results are the biggest part of the history, and they can be fetched again: the file is still there. Your task is to replace the content of all tool results except the last `keep_last` ones with a short placeholder that tells the model what to do if it needs the result. The tool calls and the structure of the conversation must stay intact.

In [ ]:
# TODO: Replace the content of old tool results with a placeholder
def clear_old_tool_results(messages: list[dict], keep_last: int = 1) -> list[dict]:
    """Returns a copy of the conversation in which only the last tool results have their full content."""
    calls = {call["id"]: call["function"] for m in messages for call in m.get("tool_calls") or []}
    tool_indices = [index for index, message in enumerate(messages) if message["role"] == "tool"]
    to_clear = set(tool_indices[: len(tool_indices) - keep_last])
    cleared = []
    for index, message in enumerate(messages):
        if index in to_clear:
            call = calls[message["tool_call_id"]]
            message = {**message, "content": f"[Result removed to save context. Call {call['name']}({call['arguments']}) again if you need it.]"}
        cleared.append(message)
    return cleared


cleared = clear_old_tool_results(session, keep_last=1)
print(f"Full history:         about {history_tokens(session)} tokens")
print(f"Without old results:  about {history_tokens(cleared)} tokens")
print(cleared[9]["content"])

That helps, and it is free – but the pasted server log and the test output are not tool results and stay in the history. For that we need a summary.

#### Summarize (condensing)

Your task is to implement `condense_history`: the old turns are replaced by a summary written by the model; the last turns are kept word by word. The summary goes to the end of the system prompt, so the conversation still starts with the system prompt and continues with the kept turns.

The quality depends on the prompt: tell the model what must survive (decisions, exact names) and what can go (file contents and logs – they can be read again). The helper `format_transcript` turns the old messages into plain text and shortens long tool results and pastes.

In [ ]:
def format_transcript(messages: list[dict], max_chars: int = 600) -> str:
    """Turns messages into a plain text transcript; long contents are shortened."""
    lines = []
    for message in messages:
        if message.get("tool_calls"):
            calls = ", ".join(f"{c['function']['name']}({c['function']['arguments']})" for c in message["tool_calls"])
            lines.append(f"ASSISTANT called {calls}")
            continue
        content = message["content"]
        if len(content) > max_chars:
            content = content[:max_chars] + f" [... {len(content) - max_chars} more characters]"
        lines.append(f"{message['role'].upper()}: {content}")
    return "\n\n".join(lines)


# TODO: Write the prompt for the summary
CONDENSE_PROMPT = """You condense the history of a conversation between a developer and a coding assistant, so that the conversation can continue with a much shorter context.

Write a summary with these sections:
## Goal
## Decisions (with the exact names, codes and values, and the reason)
## Relevant files and functions
## Done so far
## Open tasks

Rules:
- Keep exact identifiers: file, function and class names, error codes, ticket ids.
- Do not copy file contents, logs or test output - say where they can be found again instead.
- At most 250 words."""


# TODO: Implement condense_history
def condense_history(messages: list[dict], keep_last_turns: int = 2) -> list[dict]:
    """Replaces all but the last turns with a summary at the end of the system prompt."""
    first_kept = turn_starts(messages)[-keep_last_turns]
    old, recent = messages[1:first_kept], messages[first_kept:]
    response = call_with_retry(
        completion,
        model=generation_model,
        messages=[{"role": "system", "content": CONDENSE_PROMPT}, {"role": "user", "content": format_transcript(old)}],
        temperature=0.0,
    )
    summary = response.choices[0].message.content
    system_prompt = f"{messages[0]['content']}\n\n## Summary of the conversation so far\n{summary}"
    return [{"role": "system", "content": system_prompt}] + recent


condensed = condense_history(session, keep_last_turns=2)
print(condensed[0]["content"])
print(f"\nCondensed history: about {history_tokens(condensed)} tokens")

In [ ]:
ask_follow_up("Condensed history", condensed)

For comparison: the full history (the most expensive variant).

In [ ]:
ask_follow_up("Full history", session)

What did we learn?
* Condensing keeps what matters for the task and drops what can be fetched again. The **prompt** of the summary decides what survives – decisions and exact identifiers must be named explicitly.
* Clearing old tool results is free and safe for results that the model can fetch again. It pairs well with tools: *just-in-time* instead of *just-in-case*.
* Condensing rewrites the beginning of the conversation – the **prompt cache** of exercise 01 is gone afterwards. That's why harnesses condense rarely and in big steps (e.g. when 80% of the budget is used) instead of after every turn. Claude Code, for example, compacts the conversation automatically near the limit or on demand with `/compact`.
* A summary always loses details. If a fact is important, it should not depend on the summary: write it down – in the ticket, the code, a file. That's what agents do with *notes*, which we will see in the next lesson.

### Learnings
* The context window is the working memory of the model – and the harness decides what is in it. **Context engineering** is the design of this decision: what goes in, when, in which order, and what stays out.
* Every request resends the complete context. The costs of a session grow faster than its length, and long, noisy contexts make the answers worse (*context rot*).
* Order the context from **stable to dynamic** to benefit from **prompt caching**: system prompt, tools and instructions first, session information and conversation last.
* Use **progressive disclosure**: give the model an index (skills, tools, files) and let it load the details just in time.
* **Condense** long histories: clear tool results that can be fetched again, summarize the rest with a prompt that keeps decisions and exact identifiers.
* The same principles apply to tools: dozens of MCP tools with long descriptions fill the context, too. Some harnesses therefore select the tools per request or load them on demand.
* Further reading: [Thoughtworks Technology Radar – Context engineering](https://www.thoughtworks.com/radar/techniques/context-engineering)
* In the next lesson, the assistant becomes an **agent** that works on a task for many steps on its own – and long-running agents need all of these techniques, plus some more: notes as external memory and sub-agents that work in a separate context.